# Speech Emotion Recognition — train all 9 models (Google Colab T4)

**Before running**
1. Runtime → Change runtime type → **T4 GPU**.
2. Upload your Kaggle API token as `kaggle.json`, then run the auth cell.
3. Runtime → Run all. Finished `(model, dataset)` pairs are skipped on re-run.

**Open in Colab** (after pushed to GitHub):
https://colab.research.google.com/github/Sanjeev2004/Speech-Emotion-Recognization/blob/full-project/ml/notebooks/train_on_colab.ipynb

**Output:** `/content/ser_outputs.zip` → copy `export/*` into `backend/models/` and `reports/*` into `reports/`.

In [ ]:
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime → Change runtime type → T4 GPU'

In [ ]:
# Upload kaggle.json via the Colab files UI, or paste credentials once.
from pathlib import Path
import shutil, os
src = Path('/content/kaggle.json')
dst = Path.home() / '.kaggle' / 'kaggle.json'
dst.parent.mkdir(parents=True, exist_ok=True)
if src.exists():
    shutil.copy(src, dst)
    os.chmod(dst, 0o600)
assert dst.exists(), 'Upload /content/kaggle.json (from https://www.kaggle.com/settings → API → Create New Token)'
!pip -q install kaggle

In [ ]:
REPO_URL = 'https://github.com/Sanjeev2004/Speech-Emotion-Recognization.git'
BRANCH = 'full-project'
!git clone -q -b $BRANCH $REPO_URL repo 2>/dev/null || (cd repo && git fetch -q && git checkout -q $BRANCH && git pull -q)
%cd /content/repo/ml
!pip -q install -r requirements.txt

In [ ]:
import glob, os
from pathlib import Path
DATA = Path('/content/data')
DATA.mkdir(exist_ok=True)
# Official Kaggle dataset slugs used by this project
!kaggle datasets download -d uwrfkaggler/ravdess-emotional-speech-audio -p {DATA} --unzip -q
!kaggle datasets download -d ejlok1/cremad -p {DATA}/cremad --unzip -q
RAVDESS_DIR = str(DATA / 'ravdess-emotional-speech-audio') if (DATA / 'ravdess-emotional-speech-audio').exists() else str(DATA)
CREMAD_DIR = str(DATA / 'cremad')
# Some Kaggle zips unpack flat; search for Actor_* folders
if not glob.glob(RAVDESS_DIR + '/**/Actor_*', recursive=True):
    RAVDESS_DIR = str(DATA)
print(len(glob.glob(RAVDESS_DIR + '/**/*.wav', recursive=True)), 'RAVDESS wav files')
print(len(glob.glob(CREMAD_DIR + '/**/*.wav', recursive=True)), 'CREMA-D wav files')
WORK = '/content'
ARGS = f'--ravdess-dir {RAVDESS_DIR} --cremad-dir {CREMAD_DIR} --cache-dir {WORK}/cache --out {WORK}/runs --report-dir {WORK}/reports'

In [ ]:
from ser.datasets import build_manifest
df = build_manifest(RAVDESS_DIR, CREMAD_DIR)
display(df.groupby(['dataset', 'split']).size().unstack())
display(df.groupby(['dataset', 'emotion']).size().unstack())

## Classical + spectrogram models (~1 h on T4)
Includes re-runs of svm, random_forest, bilstm plus cnn2d, cnn_lstm, crnn_attention.

In [ ]:
!python -m ser.run_all $ARGS --force --models svm random_forest bilstm cnn2d cnn_lstm crnn_attention

## SSL transformers (~30–60 min each on T4)
If the session is about to disconnect, re-run from here; completed runs are skipped unless you pass `--force`.

In [ ]:
!python -m ser.run_all $ARGS --force --models wav2vec2 hubert wavlm

In [ ]:
from IPython.display import Image, Markdown, display
!python -m ser.compare --runs $WORK/runs --out $WORK/reports
display(Markdown(open(f'{WORK}/reports/leaderboard.md').read()))
display(Image(f'{WORK}/reports/uar_by_model.png'))
if os.path.exists(f'{WORK}/reports/cross_dataset.png'):
    display(Image(f'{WORK}/reports/cross_dataset.png'))

In [ ]:
import pandas as pd
from ser.models import REGISTRY
lb = pd.read_csv(f'{WORK}/reports/leaderboard.csv')
ok = lb[(lb.train_dataset == 'combined') & (lb.test_dataset == 'combined')
        & lb.model.map(lambda m: REGISTRY.get(m) and REGISTRY[m].deployable)]
if ok.empty:
    ok = lb[(lb.train_dataset == 'cremad') & (lb.test_dataset == 'cremad')
            & lb.model.map(lambda m: REGISTRY.get(m) and REGISTRY[m].deployable)]
best = ok.sort_values('uar', ascending=False).iloc[0]
run = f"{best.model}__{best.train_dataset}"
print('Deploying:', run, 'UAR', round(best.uar, 3))
!python -m ser.export_onnx --run $WORK/runs/{run} --out $WORK/export --leaderboard $WORK/reports/leaderboard.json
!cd $WORK && zip -qr ser_outputs.zip reports export
print('Download /content/ser_outputs.zip')